# Notebook 4: 

produce an excel with valuations for thermic plants

In [39]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt

In [40]:
#cargar archivos
df_result = pd.read_csv("../results/fechas_117/df_20250401_to_20250909_M20_preproc.csv") #cambiar la direccion
## calcular quienes son las firmas que entran al equilibrio!!
df_transado =pd.read_csv("../datos/procesado/df_despacho_agg.csv")
df_transado['FechaHora'] = pd.to_datetime(df_transado['FechaHora'])

#params
M=20  #ajustar
sub_dir='../results/fechas_117/' #cambiar la direccion

In [41]:
df_transado

,FechaHora,daily_eq_demand
0,2025-04-01 00:00:00,8634480
1,2025-04-01 01:00:00,8343930
2,2025-04-01 02:00:00,8149800
3,2025-04-01 03:00:00,8060810
4,2025-04-01 04:00:00,8231010
...,...,...
3883,2025-09-09 19:00:00,11822380
3884,2025-09-09 20:00:00,11697040
3885,2025-09-09 21:00:00,11140860
3886,2025-09-09 22:00:00,10420020


In [42]:
# Asegurar formato datetime
df_result['FechaHora'] = pd.to_datetime(df_result['FechaHora'])
df_transado['FechaHora'] = pd.to_datetime(df_transado['FechaHora'])

# Unir la demanda por FechaHora
df_result = df_result.merge(
    df_transado[['FechaHora', 'daily_eq_demand']],
    on='FechaHora',
    how='left'
)

# Ordenar por FechaHora y precio
df_result = df_result.sort_values(['FechaHora', 'precio'])

# Crear variable ganador iniciando en 0
df_result['ganador'] = 0

# Función que decide ganadores dentro de cada FechaHora
def asignar_ganadores(grupo):
    # Ordenar por precio ascendente
    grupo = grupo.sort_values('precio').reset_index(drop=True)

    demanda = grupo['daily_eq_demand'].iloc[0]
    suma = 0
    winners = []
    cumulatives = []

    for qty in grupo['cantidad_pos']:
        
        # El ganador es TRUE solo si la oferta completa cabe dentro de la demanda restante
        if suma + qty <= demanda:
            winners.append(1)
            suma += qty
        else:
            winners.append(0)
            suma += qty  # igual acumulamos para efectos de visualización

        cumulatives.append(suma)

    grupo['ganador'] = winners
    grupo['cumulative_qty'] = cumulatives

    return grupo


 

# Aplicar por FechaHora
df_result_gen = (
    df_result
    .groupby('FechaHora', group_keys=False)
    .apply(asignar_ganadores)
)

C:\Users\HP\AppData\Local\Temp\ipykernel_40008\544173161.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(asignar_ganadores)


In [43]:
df_result_gen

,FechaHora,CodigoPlanta,precio,cantidad,cantidad_pos,Fecha,cluster,gamma_thumb,EQpos,EdQb,daily_eq_demand,ganador,cumulative_qty
0,2025-04-01 00:00:00,HMIN,94.37,34000.00,34000.00,2025-04-01,10,5.371719,34000.00,-4.356358e+03,8634480,1,34000.00
1,2025-04-01 00:00:00,CLL1,94.37,63000.00,63000.00,2025-04-01,10,6.038973,63000.00,-4.407591e+03,8634480,1,97000.00
2,2025-04-01 00:00:00,TYP4,101.37,46000.00,46000.00,2025-04-01,10,0.528870,46000.00,-7.713285e+03,8634480,1,143000.00
3,2025-04-01 00:00:00,TYP3,101.37,49000.00,49000.00,2025-04-01,10,5.332097,49000.00,-4.397277e+03,8634480,1,192000.00
4,2025-04-01 00:00:00,TYP1,101.37,17507.26,17507.26,2025-04-01,10,0.277062,17507.26,-5.425573e+03,8634480,1,209507.26
...,...,...,...,...,...,...,...,...,...,...,...,...,...
52,2025-09-09 23:00:00,TBQ4,1117.35,60000.00,60000.00,2025-09-09,12,0.500521,60000.00,-8.662953e-25,9877640,0,13578234.65
53,2025-09-09 23:00:00,TVL1,1183.48,241000.00,241000.00,2025-09-09,12,0.995918,241000.00,-2.303208e+03,9877640,0,13819234.65
54,2025-09-09 23:00:00,TEC1,1485.40,213000.00,213000.00,2025-09-09,12,0.747557,213000.00,-1.980057e+03,9877640,0,14032234.65
55,2025-09-09 23:00:00,CTG2,1731.30,39000.00,39000.00,2025-09-09,12,0.264999,39000.00,-8.119555e+02,9877640,0,14071234.65


In [44]:
# Python (pandas): mantener solo esa fecha-hora y exportar
import pandas as pd

df_result_gen["FechaHora"] = pd.to_datetime(df_result_gen["FechaHora"])
df_out = df_result_gen[df_result_gen["FechaHora"] == pd.Timestamp("2025-04-01 00:00:00")]
df_out.to_csv("salida.csv", sep=";", decimal=",", index=False)



In [45]:
#### tipo generador
df_desglose = pd.read_excel("../datos/raw/desglose.xlsx",header=3) #cambiar la direccion
df_result_gen['Fecha'] = pd.to_datetime(df_result_gen['Fecha'])

df_result_gen = df_result_gen.merge(
    df_desglose[["Código SIC", "Tipo Generación"]],
    left_on="CodigoPlanta",
    right_on="Código SIC",
    how="left"
)
df_result_gen = df_result_gen.drop(columns=["Código SIC"])

In [46]:
df_result_gen['cantidad_pos'] = pd.to_numeric(df_result_gen['cantidad_pos'], errors='coerce')
df_result_gen

,FechaHora,CodigoPlanta,precio,cantidad,cantidad_pos,Fecha,cluster,gamma_thumb,EQpos,EdQb,daily_eq_demand,ganador,cumulative_qty,Tipo Generación
0,2025-04-01 00:00:00,HMIN,94.37,34000.00,34000.00,2025-04-01,10,5.371719,34000.00,-4.356358e+03,8634480,1,34000.00,HIDRAULICA
1,2025-04-01 00:00:00,CLL1,94.37,63000.00,63000.00,2025-04-01,10,6.038973,63000.00,-4.407591e+03,8634480,1,97000.00,HIDRAULICA
2,2025-04-01 00:00:00,TYP4,101.37,46000.00,46000.00,2025-04-01,10,0.528870,46000.00,-7.713285e+03,8634480,1,143000.00,TERMICA
3,2025-04-01 00:00:00,TYP3,101.37,49000.00,49000.00,2025-04-01,10,5.332097,49000.00,-4.397277e+03,8634480,1,192000.00,TERMICA
4,2025-04-01 00:00:00,TYP1,101.37,17507.26,17507.26,2025-04-01,10,0.277062,17507.26,-5.425573e+03,8634480,1,209507.26,TERMICA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176189,2025-09-09 23:00:00,TBQ4,1117.35,60000.00,60000.00,2025-09-09,12,0.500521,60000.00,-8.662953e-25,9877640,0,13578234.65,TERMICA
176190,2025-09-09 23:00:00,TVL1,1183.48,241000.00,241000.00,2025-09-09,12,0.995918,241000.00,-2.303208e+03,9877640,0,13819234.65,TERMICA
176191,2025-09-09 23:00:00,TEC1,1485.40,213000.00,213000.00,2025-09-09,12,0.747557,213000.00,-1.980057e+03,9877640,0,14032234.65,TERMICA
176192,2025-09-09 23:00:00,CTG2,1731.30,39000.00,39000.00,2025-09-09,12,0.264999,39000.00,-8.119555e+02,9877640,0,14071234.65,TERMICA


In [47]:
import pandas as pd

# 1. Filtrar generación NO térmica ganadora
df_no_termica = df_result_gen[
    (df_result_gen["ganador"] == 1) &
    (df_result_gen["Tipo Generación"] != "TERMICA")
]

# 2. Sumar cantidad_pos por FechaHora
no_termica_sum = (
    df_no_termica
    .groupby("FechaHora", as_index=False)["cantidad_pos"]
    .sum()
    .rename(columns={"cantidad_pos": "oferta_no_termica"})
)

# 3. Merge con el dataframe original
df_result_gen = df_result_gen.merge(
    no_termica_sum,
    on="FechaHora",
    how="left"
)

# 4. NaN → 0 (cuando no hubo oferta no térmica)
df_result_gen["oferta_no_termica"] = (
    df_result_gen["oferta_no_termica"]
    .fillna(0)
)

# 5. Demanda remanente para térmicos
df_result_gen["demanda_remanente_termica"] = (
    df_result_gen["daily_eq_demand"] - df_result_gen["oferta_no_termica"]
)

# 6. Evitar negativos
df_result_gen["demanda_remanente_termica"] = (
    df_result_gen["demanda_remanente_termica"]
    .clip(lower=0)
)



In [48]:
df_result=df_result_gen
df_result['ratio_prueba']= df_result['EQpos'] / df_result['EdQb']
df_result['Fecha'] = pd.to_datetime(df_result['Fecha'])
df_result['precio'] = pd.to_numeric(df_result['precio'], errors='coerce')


df_result['phi_h'] =  np.maximum(
    df_result['precio'] + np.where(
        df_result['EdQb'] != 0,
         (df_result['EQpos'] / df_result['EdQb']),
       0
    ),
    0
)

In [49]:
df_result

,FechaHora,CodigoPlanta,precio,cantidad,cantidad_pos,Fecha,cluster,gamma_thumb,EQpos,EdQb,daily_eq_demand,ganador,cumulative_qty,Tipo Generación,oferta_no_termica,demanda_remanente_termica,ratio_prueba,phi_h
0,2025-04-01 00:00:00,HMIN,94.37,34000.00,34000.00,2025-04-01,10,5.371719,34000.00,-4.356358e+03,8634480,1,34000.00,HIDRAULICA,8396392.03,238087.97,-7.804685e+00,86.565315
1,2025-04-01 00:00:00,CLL1,94.37,63000.00,63000.00,2025-04-01,10,6.038973,63000.00,-4.407591e+03,8634480,1,97000.00,HIDRAULICA,8396392.03,238087.97,-1.429352e+01,80.076478
2,2025-04-01 00:00:00,TYP4,101.37,46000.00,46000.00,2025-04-01,10,0.528870,46000.00,-7.713285e+03,8634480,1,143000.00,TERMICA,8396392.03,238087.97,-5.963737e+00,95.406263
3,2025-04-01 00:00:00,TYP3,101.37,49000.00,49000.00,2025-04-01,10,5.332097,49000.00,-4.397277e+03,8634480,1,192000.00,TERMICA,8396392.03,238087.97,-1.114326e+01,90.226741
4,2025-04-01 00:00:00,TYP1,101.37,17507.26,17507.26,2025-04-01,10,0.277062,17507.26,-5.425573e+03,8634480,1,209507.26,TERMICA,8396392.03,238087.97,-3.226804e+00,98.143196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176189,2025-09-09 23:00:00,TBQ4,1117.35,60000.00,60000.00,2025-09-09,12,0.500521,60000.00,-8.662953e-25,9877640,0,13578234.65,TERMICA,9054234.65,823405.35,-6.926044e+28,0.000000
176190,2025-09-09 23:00:00,TVL1,1183.48,241000.00,241000.00,2025-09-09,12,0.995918,241000.00,-2.303208e+03,9877640,0,13819234.65,TERMICA,9054234.65,823405.35,-1.046367e+02,1078.843343
176191,2025-09-09 23:00:00,TEC1,1485.40,213000.00,213000.00,2025-09-09,12,0.747557,213000.00,-1.980057e+03,9877640,0,14032234.65,TERMICA,9054234.65,823405.35,-1.075726e+02,1377.827357
176192,2025-09-09 23:00:00,CTG2,1731.30,39000.00,39000.00,2025-09-09,12,0.264999,39000.00,-8.119555e+02,9877640,0,14071234.65,TERMICA,9054234.65,823405.35,-4.803219e+01,1683.267811


In [50]:
## hasta aqui vamos bien 15 enero 2026

In [51]:
#agregar a nivel dia por empresa, esto es sumar h=1 hasta 23
# -------------------------------------------------------------------------
# Encontrar el termino  ∑_{h=0}^{23} E_{-it}[ Q_{ith} - pos_{ith} | s, p_{ht}=b_{it} ] )
# y el termino ( ∑_{h=0}^{23} E_{-it}[ ∂Q_{ith}/∂b_{it}    | s, p_{ht}=b_{it} ] )
# -------------------------------------------------------------------------

import pandas as pd
import numpy as np

df_result["FechaHora"] = pd.to_datetime(df_result["FechaHora"], errors="coerce")
df_result = df_result.dropna(subset=["FechaHora"]).copy()
df_result["Fecha"] = df_result["FechaHora"].dt.date
df_result["Hora"] = df_result["FechaHora"].dt.hour  # 0..23

# 1) Collapse duplicates within the same (Fecha, CodigoPlanta, Hora)
df_h = (
    df_result.groupby(["Fecha", "CodigoPlanta", "Hora"], as_index=False)
    .agg(
        EQpos=("EQpos", "sum"),
        EdQb=("EdQb", "sum"),
        precio=("precio", "max"),
        ganador=("ganador", "max"),
        cantidad_pos=("cantidad_pos", "sum"),
        daily_eq_demand=("daily_eq_demand", "max"),
        demanda_remanente_termica=("demanda_remanente_termica", "max"),
        cluster=('cluster', 'mean')
    )
)

# 2) Complete missing hours with a clean MultiIndex reindex (no groupby-apply)
hours = pd.Index(range(24), name="Hora")

# Create the full grid of (Fecha, CodigoPlanta, Hora)
full_index = pd.MultiIndex.from_product(
    [df_h["Fecha"].unique(), df_h["CodigoPlanta"].unique(), hours],
    names=["Fecha", "CodigoPlanta", "Hora"],
)

# Reindex onto full grid
df_h2 = (
    df_h.set_index(["Fecha", "CodigoPlanta", "Hora"])
        .reindex(full_index)
        .reset_index()
)

# 3) Fill missing offer-related values with 0
for c in ["EQpos", "EdQb", "precio", "ganador", "cantidad_pos"]:
    df_h2[c] = df_h2[c].fillna(0)

# Demand columns: within each Fecha, carry values across hours (or set to 0 if you prefer)
for c in ["daily_eq_demand", "demanda_remanente_termica"]:
    df_h2[c] = df_h2.groupby("Fecha")[c].transform(lambda s: s.ffill().bfill())

# If still missing (e.g., entire date missing demand), set to 0
df_h2["daily_eq_demand"] = df_h2["daily_eq_demand"].fillna(0)
df_h2["demanda_remanente_termica"] = df_h2["demanda_remanente_termica"].fillna(0)

# 4) Aggregate to 24-length lists in hour order, per (Fecha, CodigoPlanta)
df_grouped = (
    df_h2.sort_values(["Fecha", "CodigoPlanta", "Hora"])
        .groupby(["Fecha", "CodigoPlanta"], as_index=False)
        .agg(
            numerador=("EQpos", "sum"),
            denominador=("EdQb", "sum"),
            precio_d=("precio", "max"),
            ganador=("ganador", "max"),
            available_q=("cantidad_pos", list),                 # length 24
            daily_eq_demand=("daily_eq_demand", list),          # length 24
            daily_reman_demand=("demanda_remanente_termica", list),
            cluster=("cluster", "max")
        )
)

# 5) Assert length 24
for col in ["available_q", "daily_eq_demand", "daily_reman_demand"]:
    bad = df_grouped[col].apply(lambda x: len(x) != 24)
    if bad.any():
        raise ValueError(f"{col} has rows not length 24. Example rows: {df_grouped.loc[bad, ['Fecha','CodigoPlanta']].head()}")


In [52]:

#encontrar formula de costo marginal:
# -------------------------------------------------------------------------
#
#      φ_{it} = b_{it} +
#               ( ∑_{h=0}^{23} E_{-it}[ Q_{ith} - pos_{ith} | s, p_{ht}=b_{it} ] )
#               ----------------------------------------------------------------
#               ( ∑_{h=0}^{23} E_{-it}[ ∂Q_{ith}/∂b_{it}    | s, p_{ht}=b_{it} ] )
#
# -------------------------------------------------------------------------

# Evitar divisiones por cero
df_grouped['phi'] = np.maximum(
    df_grouped['precio_d'] + np.where(
        df_grouped['denominador'] != 0,
        df_grouped['numerador'] / df_grouped['denominador'],
        0
    ),
    0
)
df_grouped['Fecha'] = pd.to_datetime(df_grouped['Fecha'])

In [53]:
df_grouped

,Fecha,CodigoPlanta,numerador,denominador,precio_d,ganador,available_q,daily_eq_demand,daily_reman_demand,cluster,phi
0,2025-04-01,2QEK,120000.00,-63180.187366,113.37,1.0,"[5000.0, 5000.0, 5000.0, 5000.0, 5000.0, 5000....","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,111.470670
1,2025-04-01,3DDT,861305.73,-33157.180533,107.37,1.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7400.05, 58000....","[9039990.0, 9039990.0, 9039990.0, 9039990.0, 9...","[671990.0, 671990.0, 671990.0, 671990.0, 67199...",10.0,81.393553
2,2025-04-01,3ENA,4800000.00,-82282.458218,889.28,0.0,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,830.944358
3,2025-04-01,3HF5,453773.97,-28667.935925,107.37,1.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3410.36, 13129....","[9039990.0, 9039990.0, 9039990.0, 9039990.0, 9...","[671990.0, 671990.0, 671990.0, 671990.0, 67199...",10.0,91.541376
4,2025-04-01,3INX,275076.26,-27206.220316,97.37,1.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 76.26, 14000.0,...","[10333620.0, 10333620.0, 10333620.0, 10333620....","[1149620.0, 1149620.0, 1149620.0, 1149620.0, 1...",10.0,87.259214
...,...,...,...,...,...,...,...,...,...,...,...
8887,2025-09-09,URA1,3096000.00,-11575.164763,118.94,1.0,"[83000.0, 83000.0, 83000.0, 83000.0, 83000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,0.000000
8888,2025-09-09,ZPA2,468000.00,-7778.512512,372.94,1.0,"[36000.0, 36000.0, 36000.0, 36000.0, 36000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,312.774255
8889,2025-09-09,ZPA3,1260000.00,-15419.687473,346.94,1.0,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,265.226282
8890,2025-09-09,ZPA4,1280000.00,-11976.301401,360.94,1.0,"[64000.0, 64000.0, 64000.0, 64000.0, 64000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,254.062262


In [54]:
df_grouped['Fecha'] = pd.to_datetime(df_grouped['Fecha'])

df_merged = df_grouped.merge(
    df_desglose[["Código SIC", "Tipo Generación"]],
    left_on="CodigoPlanta",
    right_on="Código SIC",
    how="left"
)
df_merged = df_merged.drop(columns=["Código SIC"])

In [55]:
df_merged

,Fecha,CodigoPlanta,numerador,denominador,precio_d,ganador,available_q,daily_eq_demand,daily_reman_demand,cluster,phi,Tipo Generación
0,2025-04-01,2QEK,120000.00,-63180.187366,113.37,1.0,"[5000.0, 5000.0, 5000.0, 5000.0, 5000.0, 5000....","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,111.470670,HIDRAULICA
1,2025-04-01,3DDT,861305.73,-33157.180533,107.37,1.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7400.05, 58000....","[9039990.0, 9039990.0, 9039990.0, 9039990.0, 9...","[671990.0, 671990.0, 671990.0, 671990.0, 67199...",10.0,81.393553,SOLAR
2,2025-04-01,3ENA,4800000.00,-82282.458218,889.28,0.0,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,830.944358,TERMICA
3,2025-04-01,3HF5,453773.97,-28667.935925,107.37,1.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3410.36, 13129....","[9039990.0, 9039990.0, 9039990.0, 9039990.0, 9...","[671990.0, 671990.0, 671990.0, 671990.0, 67199...",10.0,91.541376,SOLAR
4,2025-04-01,3INX,275076.26,-27206.220316,97.37,1.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 76.26, 14000.0,...","[10333620.0, 10333620.0, 10333620.0, 10333620....","[1149620.0, 1149620.0, 1149620.0, 1149620.0, 1...",10.0,87.259214,SOLAR
...,...,...,...,...,...,...,...,...,...,...,...,...
8887,2025-09-09,URA1,3096000.00,-11575.164763,118.94,1.0,"[83000.0, 83000.0, 83000.0, 83000.0, 83000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,0.000000,HIDRAULICA
8888,2025-09-09,ZPA2,468000.00,-7778.512512,372.94,1.0,"[36000.0, 36000.0, 36000.0, 36000.0, 36000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,312.774255,TERMICA
8889,2025-09-09,ZPA3,1260000.00,-15419.687473,346.94,1.0,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,265.226282,TERMICA
8890,2025-09-09,ZPA4,1280000.00,-11976.301401,360.94,1.0,"[64000.0, 64000.0, 64000.0, 64000.0, 64000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,254.062262,TERMICA


In [56]:
df_merged = df_merged[
    (df_merged["Tipo Generación"] == "TERMICA") 
].copy()


df_merged2 = df_merged[
    (df_merged["Tipo Generación"] == "TERMICA") &
    (df_merged["ganador"] == 1)
].copy()



In [57]:

df_eliminados= df_merged[
    (df_merged["Tipo Generación"] == "TERMICA") &
    (df_merged["ganador"] == 0)
].copy()
df_eliminados.to_csv(
    "../results/fechas_117/TERMICA/EliminadosinputRed.csv",
    sep=";",
    decimal=",",
    index=False,
    float_format="%.10f"   # evita notación científica y pérdida/artefactos
)


In [58]:
#introduje esto en feb 1 2026
#df_merged= df_merged2

In [59]:
df_merged['CodigoPlanta'].unique()

array(['3ENA', 'CTG1', 'CTG2', 'GE32', 'GEC3', 'MRL1', 'PPA1', 'PPA2',
       'PPA3', 'PPA4', 'PRG1', 'PRG2', 'TBQ3', 'TBQ4', 'TBST', 'TCDT',
       'TDR1', 'TEC1', 'TFL1', 'TFL4', 'TGJ1', 'TGJ2', 'TRM1', 'TRN1',
       'TSJ1', 'TSJ2', 'TSR1', 'TVL1', 'TYP1', 'TYP2', 'TYP3', 'TYP4',
       'ZPA2', 'ZPA3', 'ZPA4', 'ZPA5'], dtype=object)

In [60]:
len(df_merged2[df_merged2['ganador']==1]['CodigoPlanta'].unique())

22

In [61]:
df_merged

,Fecha,CodigoPlanta,numerador,denominador,precio_d,ganador,available_q,daily_eq_demand,daily_reman_demand,cluster,phi,Tipo Generación
2,2025-04-01,3ENA,4800000.0,-82282.458218,889.28,0.0,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,830.944358,TERMICA
13,2025-04-01,CTG1,1248000.0,-45686.596991,1727.79,0.0,"[52000.0, 52000.0, 52000.0, 52000.0, 52000.0, ...","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,1700.473455,TERMICA
14,2025-04-01,CTG2,1368000.0,-41643.998983,1637.40,0.0,"[57000.0, 57000.0, 57000.0, 57000.0, 57000.0, ...","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,1604.550129,TERMICA
19,2025-04-01,GE32,5870000.0,-81989.545239,288.92,1.0,"[255000.0, 255000.0, 255000.0, 255000.0, 25500...","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,217.325506,TERMICA
20,2025-04-01,GEC3,3936000.0,-80245.402247,272.23,1.0,"[164000.0, 164000.0, 164000.0, 164000.0, 16400...","[8634480.0, 8343930.0, 8149800.0, 8060810.0, 8...","[238087.97000000067, 1032624.4100000001, 83845...",10.0,223.180461,TERMICA
...,...,...,...,...,...,...,...,...,...,...,...,...
8886,2025-09-09,TYP4,0.0,0.000000,0.00,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[9877640.0, 9877640.0, 9877640.0, 9877640.0, 9...","[823405.3499999996, 823405.3499999996, 823405....",NaN,0.000000,TERMICA
8888,2025-09-09,ZPA2,468000.0,-7778.512512,372.94,1.0,"[36000.0, 36000.0, 36000.0, 36000.0, 36000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,312.774255,TERMICA
8889,2025-09-09,ZPA3,1260000.0,-15419.687473,346.94,1.0,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,265.226282,TERMICA
8890,2025-09-09,ZPA4,1280000.0,-11976.301401,360.94,1.0,"[64000.0, 64000.0, 64000.0, 64000.0, 64000.0, ...","[9221580.0, 8941630.0, 8759120.0, 8656310.0, 8...","[860580.0, 580630.0, 322051.0700000003, 256310...",12.0,254.062262,TERMICA


In [62]:
df_merged_termica= df_merged[['CodigoPlanta','available_q', 'phi', 'precio_d', 'daily_reman_demand', 'Fecha', 'cluster']]

In [63]:
df_merged_termica[df_merged_termica['cluster'].isna()]

,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster
42,PRG1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[671990.0, 671990.0, 671990.0, 671990.0, 67199...",2025-04-01,NaN
43,PRG2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[671990.0, 671990.0, 671990.0, 671990.0, 67199...",2025-04-01,NaN
61,TRM1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[671990.0, 671990.0, 671990.0, 671990.0, 67199...",2025-04-01,NaN
68,TYP2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[671990.0, 671990.0, 671990.0, 671990.0, 67199...",2025-04-01,NaN
73,ZPA3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[671990.0, 671990.0, 671990.0, 671990.0, 67199...",2025-04-01,NaN
...,...,...,...,...,...,...,...
8870,TDR1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[823405.3499999996, 823405.3499999996, 823405....",2025-09-09,NaN
8872,TFL1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[823405.3499999996, 823405.3499999996, 823405....",2025-09-09,NaN
8875,TGJ2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[823405.3499999996, 823405.3499999996, 823405....",2025-09-09,NaN
8877,TRM1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,"[823405.3499999996, 823405.3499999996, 823405....",2025-09-09,NaN


In [64]:


df = df_merged_termica.copy()

# cluster a numérico (NaN si no parsea)
df["cluster"] = pd.to_numeric(df["cluster"], errors="coerce")

# Mapa por Fecha: toma daily_reman_demand de CUALQUIER fila con cluster no-NaN (primera disponible)
demand_by_date = (
    df.loc[df["cluster"].notna(), ["Fecha", "daily_reman_demand"]]
      .dropna(subset=["Fecha", "daily_reman_demand"])
      .drop_duplicates(subset=["Fecha"])
      .set_index("Fecha")["daily_reman_demand"]
)

# Rellena daily_reman_demand donde cluster es NaN usando el valor de esa Fecha
m = df["cluster"].isna()
df.loc[m, "daily_reman_demand"] = df.loc[m, "Fecha"].map(demand_by_date)

df_merged_termica = df



In [65]:
df_merged_termica[df_merged_termica['Fecha']=='2025-04-01']

,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster
2,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",830.944358,889.28,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
13,CTG1,"[52000.0, 52000.0, 52000.0, 52000.0, 52000.0, ...",1700.473455,1727.79,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
14,CTG2,"[57000.0, 57000.0, 57000.0, 57000.0, 57000.0, ...",1604.550129,1637.40,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
19,GE32,"[255000.0, 255000.0, 255000.0, 255000.0, 25500...",217.325506,288.92,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
20,GEC3,"[164000.0, 164000.0, 164000.0, 164000.0, 16400...",223.180461,272.23,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
31,MRL1,"[164000.0, 164000.0, 164000.0, 164000.0, 16400...",968.472039,1028.13,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
35,PPA1,"[21000.0, 21000.0, 21000.0, 21000.0, 21000.0, ...",290.137841,297.98,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
36,PPA2,"[60000.0, 60000.0, 60000.0, 60000.0, 60000.0, ...",256.486265,274.54,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
37,PPA3,"[70000.0, 70000.0, 70000.0, 70000.0, 70000.0, ...",264.217887,285.53,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
38,PPA4,"[165000.0, 165000.0, 165000.0, 165000.0, 16500...",231.206890,281.63,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0


In [66]:


# Asegurar tipos y orden
df_merged_termica['Fecha'] = pd.to_datetime(df_merged_termica['Fecha'])
df_merged_termica = df_merged_termica.sort_values(['CodigoPlanta', 'Fecha'])

# Columnas a imputar
cols = ['available_q', 'phi', 'precio_d']

# Qué consideramos inválido (NaN o 0)
mask_invalid = (
    df_merged_termica['cluster'].isna() &
    (
        df_merged_termica[cols].isna().any(axis=1) |
        (df_merged_termica[cols] == 0).any(axis=1)
    )
)

# Filas objetivo
df_target = df_merged_termica.loc[mask_invalid,
    ['CodigoPlanta', 'Fecha']
].copy()

# Filas válidas para buscar (precio_d > 0)
df_valid = df_merged_termica[
    df_merged_termica['precio_d'] > 0
][['CodigoPlanta', 'Fecha'] + cols].copy()

# ---------- PASADO ----------
backward = pd.merge_asof(
    df_target.sort_values('Fecha'),
    df_valid.sort_values('Fecha'),
    on='Fecha',
    by='CodigoPlanta',
    direction='backward'
)

# ---------- FUTURO ----------
forward = pd.merge_asof(
    df_target.sort_values('Fecha'),
    df_valid.sort_values('Fecha'),
    on='Fecha',
    by='CodigoPlanta',
    direction='forward'
)

# Elegir pasado si existe, si no futuro
imputed = backward.copy()
for c in cols:
    imputed[c] = backward[c].combine_first(forward[c])
for c in cols:
    df_merged_termica.loc[mask_invalid, c] = imputed[c].values

In [67]:
#df_merged_termica= df_merged_termica[df_merged_termica['cluster'].notna()]

## Hacer agrupacion para obtener 10 redes

In [68]:
df_merged_termica

,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster
2,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",830.944358,889.28,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0
78,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",826.880073,889.28,"[233350.0, 205300.0, 951070.0, 872530.0, 10147...",2025-04-02,13.0
154,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",828.001249,889.28,"[895720.0, 609450.0, 422830.0, 348240.0, 55249...",2025-04-03,7.0
230,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",695.986348,889.28,"[270253.61000000034, 466977.03000000026, 23921...",2025-04-04,2.0
306,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",767.694311,889.28,"[530630.0, 268940.0, 104170.0, 406420.0, 39726...",2025-04-07,5.0
...,...,...,...,...,...,...,...
8587,ZPA5,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...",247.517152,338.94,"[907640.0, 634130.0, 569640.0, 339450.0, 34217...",2025-09-03,15.0
8663,ZPA5,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...",193.320580,338.94,"[487560.0, 261410.0, 324070.0, 304360.0, 36979...",2025-09-04,9.0
8739,ZPA5,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...",263.392797,338.94,"[957600.0, 748720.0, 575140.0, 465820.0, 67637...",2025-09-05,2.0
8815,ZPA5,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...",139.564880,338.94,"[96490.0, 92810.0, 456530.0, 289440.0, 90450.0...",2025-09-08,5.0


In [69]:
df_merged_termica = df_merged_termica.sort_values("precio_d", ascending=True)

In [70]:
df_merged_termica

,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster
6909,TYP3,"[50000.0, 50000.0, 50000.0, 50000.0, 50000.0, ...",0.000000,11.27,"[126600.0, 77740.0, 137720.0, 521300.0, 114130...",2025-08-05,12.0
6908,TYP2,"[22000.0, 22000.0, 22000.0, 22000.0, 22000.0, ...",0.000000,11.27,"[126600.0, 77740.0, 137720.0, 521300.0, 114130...",2025-08-05,12.0
8505,TYP3,"[50000.0, 50000.0, 50000.0, 50000.0, 50000.0, ...",0.000000,91.94,"[857700.0, 845560.0, 650860.0, 553730.0, 70115...",2025-09-02,12.0
8885,TYP3,"[50000.0, 50000.0, 50000.0, 50000.0, 50000.0, ...",0.000000,91.94,"[860580.0, 580630.0, 322051.0700000003, 256310...",2025-09-09,12.0
8657,TYP3,"[50000.0, 50000.0, 50000.0, 50000.0, 50000.0, ...",0.000000,91.94,"[487560.0, 261410.0, 324070.0, 304360.0, 36979...",2025-09-04,9.0
...,...,...,...,...,...,...,...
358,TDR1,"[47000.0, 47000.0, 47000.0, 47000.0, 47000.0, ...",2284.262436,2313.83,"[530630.0, 268940.0, 104170.0, 406420.0, 39726...",2025-04-07,5.0
738,TDR1,"[47000.0, 47000.0, 47000.0, 47000.0, 47000.0, ...",2270.651300,2313.83,"[404990.0, 125600.0, 104570.0, 188840.0, 12146...",2025-04-14,5.0
130,TDR1,"[47000.0, 47000.0, 47000.0, 47000.0, 47000.0, ...",2302.160872,2313.83,"[233350.0, 205300.0, 951070.0, 872530.0, 10147...",2025-04-02,13.0
814,TDR1,"[47000.0, 47000.0, 47000.0, 47000.0, 47000.0, ...",2246.174398,2313.83,"[335950.0, 131650.0, 251700.0, 112100.0, 24931...",2025-04-15,12.0


In [71]:
df_merged_termica["id_planta"] = (
    df_merged_termica["CodigoPlanta"]
    .astype(pd.CategoricalDtype(categories=pd.unique(df_merged_termica["CodigoPlanta"]), ordered=True))
    .cat.codes + 1
)



In [72]:
df_merged_termica[df_merged_termica['CodigoPlanta']=='3ENA']

,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster,id_planta
7374,3ENA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",509.821204,609.76,"[1122511.4399999995, 1122511.4399999995, 11225...",2025-08-14,9.0,29
6842,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",626.021014,771.39,"[126600.0, 77740.0, 137720.0, 521300.0, 114130...",2025-08-05,12.0,29
4562,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",699.186858,819.04,"[213930.0, 349810.0, 149810.0, 114310.0, 31568...",2025-06-24,10.0,29
3726,3ENA,"[182000.0, 182000.0, 182000.0, 182000.0, 18200...",541.152404,819.04,"[116860.0, 164450.0, 183680.0, 168500.0, 12882...",2025-06-09,5.0,29
4866,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",680.258539,819.04,"[102790.0, 178320.0, 936180.0, 829180.0, 78456...",2025-06-30,5.0,29
...,...,...,...,...,...,...,...,...
458,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",774.545926,889.28,"[436820.0, 157630.0, 159620.0, 182750.0, 21874...",2025-04-09,14.0,29
382,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",820.165671,889.28,"[144370.0, 345990.0, 168180.0, 782160.0, 25036...",2025-04-08,11.0,29
154,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",828.001249,889.28,"[895720.0, 609450.0, 422830.0, 348240.0, 55249...",2025-04-03,7.0,29
762,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",772.509281,889.28,"[335950.0, 131650.0, 251700.0, 112100.0, 24931...",2025-04-15,12.0,29


In [73]:
df=df_merged_termica.copy()

In [74]:
df_merged_termica.to_csv(
    "../results/fechas_117/TERMICA/inputRed.csv",
    sep=";",
    decimal=",",
    index=False,
    float_format="%.10f"   # evita notación científica y pérdida/artefactos
)


In [75]:
df_merged_termica['id_planta'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36], dtype=int8)

In [76]:
df_merged_termica

,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster,id_planta
6909,TYP3,"[50000.0, 50000.0, 50000.0, 50000.0, 50000.0, ...",0.000000,11.27,"[126600.0, 77740.0, 137720.0, 521300.0, 114130...",2025-08-05,12.0,1
6908,TYP2,"[22000.0, 22000.0, 22000.0, 22000.0, 22000.0, ...",0.000000,11.27,"[126600.0, 77740.0, 137720.0, 521300.0, 114130...",2025-08-05,12.0,2
8505,TYP3,"[50000.0, 50000.0, 50000.0, 50000.0, 50000.0, ...",0.000000,91.94,"[857700.0, 845560.0, 650860.0, 553730.0, 70115...",2025-09-02,12.0,1
8885,TYP3,"[50000.0, 50000.0, 50000.0, 50000.0, 50000.0, ...",0.000000,91.94,"[860580.0, 580630.0, 322051.0700000003, 256310...",2025-09-09,12.0,1
8657,TYP3,"[50000.0, 50000.0, 50000.0, 50000.0, 50000.0, ...",0.000000,91.94,"[487560.0, 261410.0, 324070.0, 304360.0, 36979...",2025-09-04,9.0,1
...,...,...,...,...,...,...,...,...
358,TDR1,"[47000.0, 47000.0, 47000.0, 47000.0, 47000.0, ...",2284.262436,2313.83,"[530630.0, 268940.0, 104170.0, 406420.0, 39726...",2025-04-07,5.0,25
738,TDR1,"[47000.0, 47000.0, 47000.0, 47000.0, 47000.0, ...",2270.651300,2313.83,"[404990.0, 125600.0, 104570.0, 188840.0, 12146...",2025-04-14,5.0,25
130,TDR1,"[47000.0, 47000.0, 47000.0, 47000.0, 47000.0, ...",2302.160872,2313.83,"[233350.0, 205300.0, 951070.0, 872530.0, 10147...",2025-04-02,13.0,25
814,TDR1,"[47000.0, 47000.0, 47000.0, 47000.0, 47000.0, ...",2246.174398,2313.83,"[335950.0, 131650.0, 251700.0, 112100.0, 24931...",2025-04-15,12.0,25
